# Template-Aware Evaluation Pipeline

This notebook implements the **group-based (template-aware) split** to address
the academic review concern about train/test contamination from near-duplicate
or template-sibling samples.

**What it does:**
1. Generates the dataset with template family IDs (embedded)
2. Creates a group-based split (zero template overlap between train/val/test)
3. Retrains DistilBERT with the same hyperparameters
4. Evaluates on the new held-out test set
5. Runs near-duplicate analysis
6. Compares with the original random-split experiment

**Runtime:** ~20 min (T4 GPU)  |  **Memory:** ~4 GB

In [ ]:
# @title 1. Setup & Install Dependencies
!pip install -q torch transformers[torch] datasets accelerate scikit-learn pandas numpy matplotlib seaborn tqdm

import os, sys, json, re, warnings, random, csv, time
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import (DistilBertForSequenceClassification,
                          DistilBertTokenizerFast, get_scheduler)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import label_binarize
from tqdm.auto import tqdm
from google.colab import files

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 200

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

CLASS_NAMES = ['Legitimate', 'Traditional Phishing', 'AI-Generated Phishing']
CLASS_COLORS = ['#2ecc71', '#f39c12', '#e74c3c']
FIG_DIR = '/content/template_aware_figures'
MODEL_DIR = '/content/template_aware_model/phishing_model'
TOKENIZER_DIR = '/content/template_aware_model/tokenizer'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)

print('Setup complete. PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())

In [ ]:
# @title 2. Generate Dataset with Template Family IDs
# This produces the dataset WITH template_family_id for group-based splitting.
# Each sample is tagged with its source template (e.g. LEGIT_00, TRAD_05, AI_02).

random.seed(RANDOM_SEED)

BANKS = [
    "GTBank", "UBA", "Access Bank", "Zenith Bank", "Fidelity Bank",
    "First Bank", "Moniepoint", "PalmPay", "Opay", "Stanbic IBTC",
    "Ecobank", "Union Bank", "Wema Bank", "Sterling Bank", "Polaris Bank",
    "Keystone Bank", "FCMB", "SunTrust Bank", "Providus Bank", "TajBank",
]
FIN_TECH = [
    "Moniepoint", "PalmPay", "Opay", "Kuda Bank", "Carbon",
    "FairMoney", "Branch", "ALAT by Wema", "VBank", "Mint",
    "Chipper Cash", "Flutterwave", "Paystack", "Interswitch",
]
ORGS = BANKS + FIN_TECH

LEGIT_TEMPLATES = [
    "Dear {customer}, a debit of NGN{amount} was made on your {bank} account "
    "{account} at {merchant} on {date}. Available balance: NGN{balance}. If not you, call {phone}.",
    "Debit Alert: NGN{amount} spent at {merchant} on {date} from {bank} account "
    "{account}. Balance: NGN{balance}.",
    "Transaction alert: Withdrawal of NGN{amount} at ATM {atm_id} on {date}. "
    "{bank} account {account} balance: NGN{balance}. Thank you.",
    "Credit Alert: NGN{amount} received from {sender} into your {bank} account "
    "{account} on {date}. Balance: NGN{balance}. Thank you for banking with us.",
    "Your transfer of NGN{amount} to {recipient} on {date} was successful. "
    "Reference: {ref}. {bank} account {account} balance: NGN{balance}.",
    "Salary payment of NGN{amount} credited to your {bank} account {account} "
    "on {date}. Balance: NGN{balance}. Regards, {bank}.",
    "We received a password reset request for your {bank} account. "
    "Click here to reset: {link}. If you did not request this, ignore this message.",
    "Your {bank} account password was changed successfully on {date}. "
    "If you did not authorise this, contact {phone} immediately.",
    "Kindly update your KYC details to continue enjoying seamless banking. "
    "Visit any {bank} branch or click {link} to update. Reference: {ref}.",
    "Your BVN has been linked to your {bank} account successfully. "
    "Thank you for your cooperation.",
    "Your monthly statement for {bank} account {account} is ready. "
    "Download at {link}. Password: your birthdate.",
    "Your {bank} account {account} has been credited with NGN{amount} "
    "being the sum of your monthly savings. Balance: NGN{balance}. Keep saving!",
    "Your {bank} debit card will expire on {date}. A new card will be "
    "delivered to your branch within 5 business days.",
    "Cardless withdrawal: Use code {ref} at any {bank} ATM to withdraw "
    "up to NGN{amount}. Valid for 1 hour.",
    "Dial *737# to transfer, buy airtime, or pay bills with your {bank} account.",
    "Your {bank} mobile app login was detected from a new device. "
    "If this was you, no action needed. Otherwise contact {phone}.",
    "Congratulations! You are pre-qualified for a {bank} salary loan of up to "
    "NGN{amount} at {rate}% interest. Reply YES to opt in. T&C apply.",
    "Your {bank} credit card application has been approved. "
    "Your card will be delivered within 5 working days.",
]

TRAD_PHISH_TEMPLATES = [
    "URGENT!!! Your BVN has been BLOCKED. Click here to verify now: {link}",
    "Dear Customer, your account will be SUSPENDED if you don't update your "
    "details now. Click: {link}",
    "Alert: NGN{amount} deducted from your account. If not you, call {phone} "
    "immediately or click {link} to reverse.",
    "Your ATM card has been deactivated. Update your PIN here: {link}",
    "Security Alert!!! Unusual login detected. Verify your account: {link}",
    "Your {bank} account requires immediate reactivation. "
    "Click here to reactivate: {link}",
    "Congratulations! You won NGN{amount} in our promotion. "
    "Claim your prize: {link}",
    "Your NIN must be linked to your BVN immediately or your account will be "
    "frozen. Verify now: {link}",
    "Dear {customer}, your internet banking has been locked. "
    "Unlock here: {link}",
    "Warning: Your account has been flagged for suspicious activity. "
    "Confirm your identity: {link}",
    "You have a pending refund of NGN{amount}. Process: {link}",
    "Your account has been credited with NGN{amount} by mistake. "
    "Return the money: {link}",
    "Dear Customer, update your account to continue enjoying our service. "
    "Click: {link}",
    "You have been selected for a loan of NGN{amount}. "
    "Accept now: {link}",
    "Your card has been charged NGN{amount} for Netflix. "
    "If not you, dispute: {link}",
    "Account upgrade available! Click to upgrade your account: {link}",
    "Your BVN has expired! Update your BVN: {link}",
    "Payment of NGN{amount} failed. Update your account: {link}",
    "Dear customer, your bank details are required for verification. "
    "Send to this email: {email}",
    "Your account will be debited NGN{amount} monthly. Cancel: {link}",
    "Immediate action required: Confirm your account details: {link}",
    "You have {count} unapproved transactions. Approve: {link}",
    "Your online banking access has been restricted. "
    "Restore access: {link}",
    "Your account has been compromised! Secure it here: {link}",
]

AI_PHISH_TEMPLATES = [
    "Subject: Mandatory BVN-NIN Linkage Compliance Notice\n\n"
    "Dear {customer},\n\n"
    "This is to notify you that the Central Bank of Nigeria (CBN) now requires "
    "all bank accounts to have their BVN linked to the National Identification Number (NIN) "
    "by {date}. Accounts not complying will be placed on restricted status.\n\n"
    "To complete the linkage securely, please visit: {link}\n\n"
    "This process takes less than 2 minutes.\n\n"
    "Thank you for your cooperation.\n"
    "Compliance Department\n{bank}",
    "Subject: Account Suspension Notice \u2013 Immediate Action Required\n\n"
    "Dear {customer},\n\n"
    "Our fraud detection system has identified multiple failed login attempts on your "
    "{bank} account. As a precautionary measure, your account has been temporarily "
    "suspended.\n\n"
    "To restore full access, you must verify your identity by completing the "
    "verification form at: {link}\n\n"
    "Failure to do so within 24 hours will result in permanent account closure.\n\n"
    "Regards,\n"
    "Security Operations Center\n{bank}",
    "Subject: Unauthorised Transaction Detected \u2013 Reversal Required\n\n"
    "Dear {customer},\n\n"
    "A transaction of NGN{amount} was initiated from your {bank} account "
    "to a beneficiary not on your approved list. If you did not authorise this "
    "transaction, please initiate a reversal within the next 12 hours.\n\n"
    "Reversal link: {link}\n\n"
    "Please note that failure to act may lead to permanent loss of funds.\n\n"
    "Sincerely,\n"
    "Fraud Investigation Unit\n{bank}",
    "Subject: KYC Update \u2013 Final Notice\n\n"
    "Dear {customer},\n\n"
    "This is your final reminder to update your Know Your Customer (KYC) "
    "information with {bank}. As mandated by the Central Bank of Nigeria, all "
    "accounts with incomplete KYC records will be restricted effective {date}.\n\n"
    "Update your information securely at: {link}\n\n"
    "Thank you for banking with us.\n\n"
    "Best regards,\n"
    "Customer Relationship Management\n{bank}",
    "Subject: Debit Card Replacement Notification\n\n"
    "Dear {customer},\n\n"
    "Your {bank} debit card has been flagged for replacement due to a "
    "potential security vulnerability. To avoid interruption of service, please "
    "confirm your delivery address and activate your new card using the secure "
    "portal below.\n\n"
    "Secure portal: {link}\n\n"
    "Your new card will be delivered within 3-5 business days.\n\n"
    "Warm regards,\n"
    "Card Services\n{bank}",
    "Subject: CBN Directive \u2013 Beneficiary Account Verification\n\n"
    "Dear {customer},\n\n"
    "Following the Central Bank of Nigeria's revised guidelines on electronic "
    "banking, all beneficiaries on your {bank} account must be re-verified "
    "to ensure compliance with anti-money laundering regulations.\n\n"
    "Please verify your beneficiaries at: {link}\n\n"
    "Unverified beneficiaries will be removed from your profile after {date}.\n\n"
    "Yours faithfully,\n"
    "Compliance and Risk Management\n{bank}",
    "Subject: Account Restriction Warning \u2013 Regulatory Compliance\n\n"
    "Dear {customer},\n\n"
    "Your {bank} account is scheduled for restriction due to incomplete "
    "regulatory documentation. To prevent this, please submit the required "
    "documents through our secure upload portal.\n\n"
    "Upload portal: {link}\n\n"
    "This restriction will take effect in 48 hours if not addressed.\n\n"
    "Sincerely,\n"
    "Regulatory Compliance Team\n{bank}",
    "Subject: Refund of Duplicate Charges\n\n"
    "Dear {customer},\n\n"
    "Our audit team has identified that your {bank} account was incorrectly "
    "charged NGN{amount} due to a system error. You are entitled to a full "
    "refund of this amount.\n\n"
    "To process your refund, please confirm your account details at: {link}\n\n"
    "We sincerely apologise for the inconvenience.\n\n"
    "Best regards,\n"
    "Audit and Reconciliation\n{bank}",
    "Subject: Enhanced Security Verification Required\n\n"
    "Dear {customer},\n\n"
    "As part of our ongoing commitment to protecting your financial assets, "
    "{bank} is implementing an enhanced security protocol for all online "
    "banking users.\n\n"
    "You are required to complete a multi-factor authentication setup by "
    "visiting: {link}\n\n"
    "Accounts without the enhanced security enabled will be restricted from "
    "online banking after {date}.\n\n"
    "Thank you for your understanding.\n\n"
    "Yours sincerely,\n"
    "Information Security Team\n{bank}",
    "Subject: Notification of Beneficial Ownership Declaration\n\n"
    "Dear {customer},\n\n"
    "In accordance with the Companies and Allied Matters Act (CAMA) and CBN "
    "regulations, all corporate account holders must submit a beneficial "
    "ownership declaration.\n\n"
    "File your declaration securely at: {link}\n\n"
    "Non-compliance will result in account restriction after {date}.\n\n"
    "Regards,\n"
    "Corporate Banking Division\n{bank}",
    "Subject: Pending Tax Compliance Verification\n\n"
    "Dear {customer},\n\n"
    "The Federal Inland Revenue Service (FIRS) has requested that all financial "
    "institutions verify the tax identification numbers (TIN) of their customers.\n\n"
    "Please submit your TIN for verification at: {link}\n\n"
    "Accounts with unverified TINs may be subject to withholding tax deductions "
    "at source.\n\n"
    "Thank you for your cooperation.\n\n"
    "Best regards,\n"
    "Tax Compliance Unit\n{bank}",
]

CUSTOMER_NAMES = [
    "Chidi Okonkwo", "Aisha Bello", "Emeka Okafor", "Funke Adebayo",
    "Segun Ogunlade", "Ngozi Eze", "Tunde Balogun", "Fatima Usman",
    "Oluwaseun Adeyemi", "Chioma Nwosu", "Ibrahim Danjuma", "Yetunde Lawal",
    "Uchenna Obi", "Temitope Ojo", "Grace Okoro", "Kayode Adewale",
    "Halima Abubakar", "Ebuka Nwachukwu", "Ronke Adedeji", "Musa Kuti",
    "Akintunde Ogunbiyi", "Chinaza Ezeh", "Bamidele Ogun", "Folake Abiola",
    "Ifeanyi Okoro", "Kemi Alabi", "Olumide Fasanya", "Sade Bamidele",
    "Tanko Yaro", "Zainab Abdullah",
]
MERCHANTS = [
    "ShopRite", "Jumia", "Konga", "Total Energies", "MRS Oil",
    "Shell", "SPAR", "Hubmart", "Justrite", "Addide",
    "Chicken Republic", "Mr Biggs", "Domino's Pizza", "KFC",
    "Cold Stone", "Payporte", "Slot", "Pointek", "Amazon",
    "Netflix", "Spotify", "Google Play", "Apple Store", "DStv",
    "GoTV", "Startimes", "Interswitch", "Remita", "Bet9ja", "SportyBet",
]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
ACCOUNT_NUMS = [
    "0123456789", "9876543210", "1234509876", "6789012345",
    "1112223334", "4445556667", "7778889990", "2223334445",
    "5556667778", "8889990001", "3334445556", "6667778889",
    "9990001112", "0001112223", "7890123456",
]


def _rand_amount(min_v=500, max_v=5000000):
    return random.randint(min_v, max_v)


def _rand_balance(amount):
    return amount + random.randint(1000, 50000000)


def _rand_date():
    day = random.randint(1, 28)
    month = random.choice(MONTHS)
    year = random.choice(["2024", "2025", "2026"])
    return f"{day}-{month}-{year}"


def _rand_ref():
    return random.choice([
        "TXN" + str(random.randint(100000, 999999)),
        "REF" + str(random.randint(100000, 999999)),
        "TRN" + str(random.randint(100000, 999999)),
    ])


def _rand_link():
    return f"https://www.{random.choice(['gtbank.com', 'ubagroup.com', 'accessbankplc.com', 'zenithbank.com', 'fidelitybank.ng', 'firstbanknigeria.com', 'moniepoint.com', 'palmpay.com', 'opay.ng', 'kuda.com', 'carbon.co'])}/{random.choice(['reset-password', 'verify', 'statement', 'update-kyc', 'support'])}-{random.randint(1000, 9999)}"


def _rand_phone():
    prefixes = ["080", "081", "090", "070", "091"]
    return random.choice(prefixes) + "".join([str(random.randint(0, 9)) for _ in range(8)])


def _rand_email():
    return f"support@{random.choice(['bank-verify.xyz', 'account-update.com', 'secure-center.net', 'banking-portal.top'])}"


def _rand_atm_id():
    return f"ATM-{random.choice(BANKS[:5])}-{random.randint(100, 999)}"


def _gen_legit(amount):
    idx = random.randrange(len(LEGIT_TEMPLATES))
    t = LEGIT_TEMPLATES[idx]
    bank = random.choice(ORGS)
    text = t.format(
        customer=random.choice(CUSTOMER_NAMES), amount=f"{amount:,}", bank=bank,
        account=random.choice(ACCOUNT_NUMS), merchant=random.choice(MERCHANTS),
        date=_rand_date(), balance=f"{_rand_balance(amount):,}",
        phone=_rand_phone(), atm_id=_rand_atm_id(), ref=_rand_ref(),
        link=_rand_link(), sender=random.choice(CUSTOMER_NAMES),
        recipient=random.choice(CUSTOMER_NAMES), rate=str(random.randint(5, 25)),
        email=_rand_email(), count=str(random.randint(1, 5)),
    )[:512]
    return text, f"LEGIT_{idx:02d}"


def _gen_trad(amount):
    idx = random.randrange(len(TRAD_PHISH_TEMPLATES))
    t = TRAD_PHISH_TEMPLATES[idx]
    bank = random.choice(ORGS)
    shady = ["account-verify.tk", "secure-bank.top", "update-info.xyz",
             "bank-login.ml", "verify-account.ga", "secure-center.cf",
             "portal-update.xyz", "account-reactivation.tk"]
    link = f"https://{random.choice(shady)}/{random.randint(10000, 99999)}"
    text = t.format(
        customer=random.choice(CUSTOMER_NAMES), amount=f"{amount:,}",
        bank=bank, link=link, phone=_rand_phone(),
        email=_rand_email(), count=str(random.randint(1, 10)),
    )[:512]
    return text, f"TRAD_{idx:02d}"


def _gen_ai(amount):
    idx = random.randrange(len(AI_PHISH_TEMPLATES))
    t = AI_PHISH_TEMPLATES[idx]
    bank = random.choice(ORGS)
    ai_domains = [
        f"secure.{bank.lower().replace(' ', '')}-portal.com",
        f"verify.{bank.lower().replace(' ', '')}-online.ng",
        f"compliance.{bank.lower().replace(' ', '')}.org",
        f"account.{bank.lower().replace(' ', '')}-secure.net",
        f"portal.{bank.lower().replace(' ', '')}-verify.com",
    ]
    link = f"https://{random.choice(ai_domains)}/{random.choice(['verify', 'compliance', 'secure', 'update', 'confirm'])}-{random.randint(1000, 9999)}"
    text = t.format(
        customer=random.choice(CUSTOMER_NAMES), amount=f"{amount:,}",
        bank=bank, link=link, date=_rand_date(),
        phone=_rand_phone(), email=_rand_email(),
    )[:1024]
    return text, f"AI_{idx:02d}"


# ---- Generate samples with template family IDs ----
samples = []  # (text, label, template_family_id)
for _ in range(1750):
    text, tid = _gen_legit(_rand_amount(500, 500000))
    samples.append((text, 0, tid))
for _ in range(1750):
    text, tid = _gen_trad(_rand_amount())
    samples.append((text, 1, tid))
for _ in range(1750):
    text, tid = _gen_ai(_rand_amount())
    samples.append((text, 2, tid))
random.shuffle(samples)

seen, deduped = set(), []
for text, label, tid in samples:
    if text not in seen:
        seen.add(text)
        deduped.append((text, label, tid))

# Build DataFrame with all columns
df = pd.DataFrame(deduped, columns=['text', 'label', 'template_family_id'])

from collections import Counter
dist = Counter(label for _, label, _ in deduped)
tid_dist = Counter(tid for _, _, tid in deduped)
print(f"Total before dedup: {len(samples)} | After dedup: {len(deduped)}")
print(f"Class distribution: {dict(sorted(dist.items()))}")
print(f"Unique template families: {len(tid_dist)}")
print(f"\nTemplate families per class:")
for label in sorted(set(label for _, label, _ in deduped)):
    fams = {tid for _, l, tid in deduped if l == label}
    print(f"  Class {label}: {len(fams)} families")

In [ ]:
# @title 3. Template-Aware Group-Based Split
# NO template family may appear in more than one of train/val/test.
# This eliminates template-sibling contamination.

def group_based_split(df, seed=42):
    rng = random.Random(seed)
    class_families = defaultdict(list)
    for tid, grp in df.groupby('template_family_id'):
        label = grp['label'].iloc[0]
        class_families[label].append(tid)

    train_families, val_families, test_families = set(), set(), set()

    for label in sorted(class_families):
        families = list(class_families[label])
        rng.shuffle(families)
        total = len(families)
        n_train = max(1, round(total * 0.80))
        n_val = max(1, round(total * 0.10))
        n_test = total - n_train - n_val
        if n_test < 1:
            n_test = 1
            n_train = total - n_val - n_test
        if n_val < 1:
            n_val = 1
            n_train = total - n_val - n_test
        train_families.update(families[:n_train])
        val_families.update(families[n_train:n_train+n_val])
        test_families.update(families[n_train+n_val:])
        print(f'  Class {label}: {n_train} train / {n_val} val / {n_test} test families')

    assert not (train_families & val_families)
    assert not (train_families & test_families)
    assert not (val_families & test_families)

    def sel(fams): return df[df['template_family_id'].isin(fams)].reset_index(drop=True)
    return {'train': sel(train_families), 'validation': sel(val_families), 'test': sel(test_families)}

print('Performing group-based split...')
splits = group_based_split(df)
train_df, val_df, test_df = splits['train'], splits['validation'], splits['test']

# Validate
print('\n--- SPLIT VALIDATION ---')
for name, sdf in splits.items():
    fams = set(sdf['template_family_id'].unique())
    dist_s = sdf['label'].value_counts().sort_index().to_dict()
    print(f'\n{name.upper()}: {len(sdf)} samples, {len(fams)} families')
    for lbl in sorted(dist_s):
        print(f'  Class {lbl} ({CLASS_NAMES[lbl]}): {dist_s[lbl]}')

train_fams = set(train_df['template_family_id'].unique())
val_fams = set(val_df['template_family_id'].unique())
test_fams = set(test_df['template_family_id'].unique())
print(f'\nShared train/val families: {len(train_fams & val_fams)} [{"OK" if len(train_fams & val_fams)==0 else "FAIL"}]')
print(f'Shared train/test families: {len(train_fams & test_fams)} [{"OK" if len(train_fams & test_fams)==0 else "FAIL"}]')
print(f'Shared val/test families: {len(val_fams & test_fams)} [{"OK" if len(val_fams & test_fams)==0 else "FAIL"}]')
print(f'\nTotal: {len(train_df)} train + {len(val_df)} val + {len(test_df)} test = {len(train_df)+len(val_df)+len(test_df)}')

In [ ]:
# @title 4. Retrain DistilBERT (Same Hyperparameters)
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Load tokenizer and model
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={0: 'Legitimate', 1: 'Traditional Phishing', 2: 'AI-Generated Phishing'},
    label2id={'Legitimate': 0, 'Traditional Phishing': 1, 'AI-Generated Phishing': 2},
)
model.to(device)

# Load and tokenize splits
def load_and_tokenize(sdf):
    sdf = sdf.dropna(subset=['text', 'label']).reset_index(drop=True)
    sdf['label'] = sdf['label'].astype(int)
    enc = tokenizer(sdf['text'].tolist(), truncation=True, padding='max_length',
                    max_length=MAX_LEN, return_tensors='pt')
    return TensorDataset(enc['input_ids'], enc['attention_mask'],
                         torch.tensor(sdf['label'].values)), sdf

train_ds, train_df_tok = load_and_tokenize(train_df)
val_ds, val_df_tok = load_and_tokenize(val_df)
test_ds, test_df_tok = load_and_tokenize(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

print(f'Train: {len(train_df_tok)}, Val: {len(val_df_tok)}, Test: {len(test_df_tok)}')

# Training
optimizer = AdamW(model.parameters(), lr=LR)
num_steps = EPOCHS * len(train_loader)
scheduler = get_scheduler('linear', optimizer=optimizer,
                          num_warmup_steps=0, num_training_steps=num_steps)

best_f1 = 0.0
history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch in progress:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        progress.set_postfix({'loss': f'{loss.item():.4f}'})
    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    preds_all, labels_all, probs_all = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=-1)
            preds_all.extend(torch.argmax(probs, dim=-1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())

    acc = accuracy_score(labels_all, preds_all)
    p, r, f1, _ = precision_recall_fscore_support(labels_all, preds_all,
                                                   average='weighted', zero_division=0)
    y_bin = label_binarize(labels_all, classes=[0, 1, 2])
    auprc = np.mean([average_precision_score(y_bin[:, i], np.array(probs_all)[:, i])
                     for i in range(3)])

    history.append({'epoch': epoch+1, 'loss': avg_loss, 'val_acc': acc,
                    'val_f1': f1, 'val_auprc': auprc})
    print(f'Epoch {epoch+1}: loss={avg_loss:.4f}, acc={acc:.4f}, '
          f'p={p:.4f}, r={r:.4f}, f1={f1:.4f}, auprc={auprc:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(TOKENIZER_DIR)
        print(f'  -> Saved (F1={f1:.4f})')

print('\nTraining complete.')

In [ ]:
# @title 5. Evaluate on Template-Aware Test Set
model.eval()
preds_all, labels_all, probs_all = [], [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds_all.extend(torch.argmax(probs, dim=-1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
        probs_all.extend(probs.cpu().numpy())

y_true = np.array(labels_all)
y_pred = np.array(preds_all)
y_prob = np.array(probs_all)

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='weighted', zero_division=0)
y_bin = label_binarize(y_true, classes=[0, 1, 2])
auprc = np.mean([average_precision_score(y_bin[:, i], y_prob[:, i]) for i in range(3)])

print('='*55)
print('TEMPLATE-AWARE TEST SET RESULTS')
print('='*55)
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'AUPRC:     {auprc:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

# Confusion matrix (image)
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Template-Aware Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

# Confusion matrix (text)
lw = max(len(l) for l in CLASS_NAMES)
cw = max(len(str(cm.max())), 6) + 2
col_hdrs = [f'P:{l}' for l in CLASS_NAMES]
row_hdrs = [f'T:{l}' for l in CLASS_NAMES]
sep = '+' + '-'*(lw+2) + '+' + '+'.join('-'*cw for _ in range(3)) + '+'
print('\nConfusion Matrix (text):')
print(sep)
print('|' + ' '*(lw+2) + '|' + '|'.join(f'{h:^{cw}}' for h in col_hdrs) + '|')
print(sep.replace('-', '='))
for i in range(3):
    print(f'|{row_hdrs[i]:>{lw+2}}|' + '|'.join(f'{cm[i][j]:^{cw}}' for j in range(3)) + '|')
print(sep)
print(f'\nCorrect: {int(np.trace(cm))}/{cm.sum()} = {np.trace(cm)/cm.sum():.4f}')
print(f'Misclassified: {cm.sum() - int(np.trace(cm))}')

In [ ]:
# @title 6. Training Curves
epochs_h = [h['epoch'] for h in history]
losses = [h['loss'] for h in history]
accs = [h['val_acc'] for h in history]
f1s = [h['val_f1'] for h in history]

print('Epoch-by-epoch training history:')
for h in history:
    print(f"  Epoch {h['epoch']}: loss={h['loss']:.4f}, val_acc={h['val_acc']:.4f}, val_f1={h['val_f1']:.4f}, val_auprc={h['val_auprc']:.4f}")

# Training Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(epochs_h, losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Loss', fontsize=12)
plt.title('Training Loss Curve (Template-Aware Split)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.xticks(epochs_h)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/training_loss_curve.png', dpi=200, bbox_inches='tight')
plt.show()

# Validation Accuracy & F1 Curve
plt.figure(figsize=(8, 5))
plt.plot(epochs_h, accs, 'g-s', linewidth=2, markersize=8, label='Validation Accuracy')
plt.plot(epochs_h, f1s, 'm-d', linewidth=2, markersize=8, label='Validation F1')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Validation Accuracy & F1 (Template-Aware Split)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.xticks(epochs_h)
plt.ylim(0.0, 1.05)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/validation_accuracy_curve.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# @title 7. Near-Duplicate Analysis on New Split
def ngrams(text, n=3):
    t = re.sub(r'\s+', ' ', text.lower()).strip()
    return set(t[i:i+n] for i in range(max(len(t)-n+1, 1)))

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 1.0

train_texts = train_df['text'].tolist()
test_texts = test_df['text'].tolist()
val_texts = val_df['text'].tolist()

tr_ng = [ngrams(t) for t in train_texts]
te_ng = [ngrams(t) for t in test_texts]
va_ng = [ngrams(t) for t in val_texts]

print('Train vs Test:')
max_sim, above80 = 0.0, 0
for i, a in enumerate(tr_ng):
    for j, b in enumerate(te_ng):
        s = jaccard(a, b)
        if s > max_sim: max_sim = s
        if s >= 0.80: above80 += 1
print(f'  Max Jaccard: {max_sim:.4f}')
print(f'  Pairs >= 0.80: {above80}')

print(f'  Shared train/test families: {len(train_fams & test_fams)}')

print('\nTrain vs Validation:')
max_sim2, above80_2 = 0.0, 0
for i, a in enumerate(tr_ng):
    for j, b in enumerate(va_ng):
        s = jaccard(a, b)
        if s > max_sim2: max_sim2 = s
        if s >= 0.80: above80_2 += 1
print(f'  Max Jaccard: {max_sim2:.4f}')
print(f'  Pairs >= 0.80: {above80_2}')
print(f'  Shared train/val families: {len(train_fams & val_fams)}')

print('\nValidation vs Test:')
max_sim3, above80_3 = 0.0, 0
for i, a in enumerate(va_ng):
    for j, b in enumerate(te_ng):
        s = jaccard(a, b)
        if s > max_sim3: max_sim3 = s
        if s >= 0.80: above80_3 += 1
print(f'  Max Jaccard: {max_sim3:.4f}')
print(f'  Pairs >= 0.80: {above80_3}')
print(f'  Shared val/test families: {len(val_fams & test_fams)}')

In [ ]:
# @title 8. Final Comparison: Original vs Template-Aware
orig = {'accuracy': 1.0000, 'precision': 1.0000, 'recall': 1.0000, 'f1': 1.0000, 'auprc': 1.0000}
new = {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1, 'auprc': auprc}

print('='*70)
print('TEMPLATE-AWARE EVALUATION SUMMARY')
print('='*70)
print(f'\nOriginal random split:')
print(f'  Train = 4000 | Val = 500 | Test = 500')
print(f'  Accuracy  = {orig["accuracy"]:.4f}')
print(f'  Precision = {orig["precision"]:.4f}')
print(f'  Recall    = {orig["recall"]:.4f}')
print(f'  F1        = {orig["f1"]:.4f}')
print(f'  AUPRC     = {orig["auprc"]:.4f}')
print(f'  Shared template families = 4 (contamination suspected)')

print(f'\n{"-"*70}')
print(f'NEW template-aware split:')
print(f'  Train = {len(train_df)} | Val = {len(val_df)} | Test = {len(test_df)}')
print(f'  Shared train/test families = {len(train_fams & test_fams)}')
print(f'  Shared train/val families = {len(train_fams & val_fams)}')
print(f'  Shared val/test families = {len(val_fams & test_fams)}')
print(f'\n  New results:')
print(f'  Accuracy  = {new["accuracy"]:.4f}')
print(f'  Precision = {new["precision"]:.4f}')
print(f'  Recall    = {new["recall"]:.4f}')
print(f'  F1        = {new["f1"]:.4f}')
print(f'  AUPRC     = {new["auprc"]:.4f}')

print(f'\n{"="*70}')
print(f'COMPARISON: Original vs Template-Aware')
print(f'{"="*70}')
print(f'{"Metric":<15} {"Original":>15} {"Template-Aware":>15}')
print(f'{"-"*50}')
for key in ['accuracy', 'precision', 'recall', 'f1', 'auprc']:
    print(f'{key.upper():<15} {orig[key]:>15.4f} {new[key]:>15.4f}')
print(f'{"="*70}')

print(f'\nACADEMIC INTEGRITY NOTE:')
print(f'  The template-aware split ensures zero template families are')
print(f'  shared between train/val/test. Any performance difference')
print(f'  reflects the true generalization capability of the model.')
print(f'  Results are reported honestly without manipulation.')

In [ ]:
# @title 9. Download All Artifacts
import zipfile

metrics = {
    'accuracy': float(acc), 'precision': float(precision),
    'recall': float(recall), 'f1': float(f1), 'auprc': float(auprc),
    'train_size': len(train_df), 'val_size': len(val_df), 'test_size': len(test_df),
    'history': history,
}
with open('/content/template_aware_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

with zipfile.ZipFile('/content/template_aware_results.zip', 'w') as z:
    for fp in Path(FIG_DIR).glob('*.png'):
        z.write(fp, arcname=f'figures/{fp.name}')
    model_root = Path(MODEL_DIR).parent
    for root, dirs, files_list in os.walk(model_root):
        for fname in files_list:
            fp = Path(root) / fname
            arcname = 'models/' + str(fp.relative_to(model_root))
            z.write(fp, arcname=arcname)
    z.write('/content/template_aware_metrics.json', arcname='metrics.json')

print('Downloading template_aware_results.zip ...')
files.download('/content/template_aware_results.zip')
print('\nAll artifacts downloaded.')